In [1]:
import polars as pl
import nflreadpy as nfl

playbyplay = nfl.load_pbp(2025)

## Q1 Scope check: 
Which teams and games are in the frame

In [2]:
teams = set(playbyplay["home_team"].to_list()) | set(playbyplay["away_team"].to_list())
print(len(teams))
sorted(teams)

32


['ARI',
 'ATL',
 'BAL',
 'BUF',
 'CAR',
 'CHI',
 'CIN',
 'CLE',
 'DAL',
 'DEN',
 'DET',
 'GB',
 'HOU',
 'IND',
 'JAX',
 'KC',
 'LA',
 'LAC',
 'LV',
 'MIA',
 'MIN',
 'NE',
 'NO',
 'NYG',
 'NYJ',
 'PHI',
 'PIT',
 'SEA',
 'SF',
 'TB',
 'TEN',
 'WAS']

In [3]:
pl.Config.set_tbl_rows(40)
reg = (
    playbyplay.select(["game_id", "season_type", "home_team", "away_team"])
    .unique()
    .filter(pl.col("season_type") == "REG")
)
per_team = (
    pl.concat([
        reg.select(pl.col("home_team").alias("team")),
        reg.select(pl.col("away_team").alias("team")),
    ])
    .group_by("team").len().sort("len")
)
per_team

team,len
str,u32
"""HOU""",17
"""BAL""",17
"""LAC""",17
"""MIA""",17
"""MIN""",17
"""CLE""",17
"""CIN""",17
"""TEN""",17
"""SEA""",17


## Q2 Scope Check: Fourth-Down Decisions

Identify all fourth-down plays recorded during the 2025 NFL regular season.

In [4]:
q2_access_frame = playbyplay.filter(
    (pl.col("season_type") == "REG") &
    (pl.col("down") == 4)
)

print("Total fourth-down plays:", q2_access_frame.height)
print("Games represented:", q2_access_frame["game_id"].n_unique())
print("Offensive teams represented:",
      q2_access_frame["posteam"].drop_nulls().n_unique())

Total fourth-down plays: 4081
Games represented: 272
Offensive teams represented: 32


In [5]:
q2_access_frame.group_by("play_type").agg(
    pl.len().alias("count")
).sort("count", descending=True)

play_type,count
str,u32
"""punt""",1933
"""field_goal""",978
"""pass""",561
"""run""",321
"""no_play""",284
"""qb_kneel""",4


In [6]:
q2_relevant = q2_access_frame.filter(
    pl.col("play_type").is_in(["run", "pass", "punt"])
)

print("Relevant fourth-down plays:", q2_relevant.height)

print("Missing yards to go:",
      q2_relevant["ydstogo"].null_count())

print("Missing EPA:",
      q2_relevant["epa"].null_count())

print("Missing field position:",
      q2_relevant["yardline_100"].null_count())

Relevant fourth-down plays: 2815
Missing yards to go: 0
Missing EPA: 0
Missing field position: 0


In [7]:
q2_sample = q2_relevant.filter(
    pl.col("ydstogo").is_not_null() &
    pl.col("epa").is_not_null()
)

print("Access frame size:", q2_access_frame.height)
print("Relevant play types:", q2_relevant.height)
print("Final EDA sample size:", q2_sample.height)

print("Excluded due to play type:",
      q2_access_frame.height - q2_relevant.height)

print("Excluded due to missing values:",
      q2_relevant.height - q2_sample.height)

Access frame size: 4081
Relevant play types: 2815
Final EDA sample size: 2815
Excluded due to play type: 1266
Excluded due to missing values: 0


## Q3 Scope Check: Wide Receiver Separation

Verify the number of wide receiver weekly observations available in the
2025 NFL Next Gen Stats receiving dataset.

In [8]:
# Load the raw Next Gen Stats receiving dataset
ngs_receiving = nfl.load_nextgen_stats(
    seasons=2025,
    stat_type="receiving"
)

print("Total raw observations:", ngs_receiving.height)
print("Total columns:", ngs_receiving.width)

Total raw observations: 1402
Total columns: 23


In [9]:
q3_access_frame = ngs_receiving.filter(
    (pl.col("season") == 2025) &
    (pl.col("season_type") == "REG") &
    (pl.col("player_position") == "WR") &
    (pl.col("week") > 0)
)

print("Q3 access frame size:", q3_access_frame.height)

print(
    "Unique wide receivers:",
    q3_access_frame["player_gsis_id"].drop_nulls().n_unique()
)

print(
    "Teams represented:",
    q3_access_frame["team_abbr"].drop_nulls().n_unique()
)

print(
    "Weeks represented:",
    q3_access_frame["week"].drop_nulls().n_unique()
)

Q3 access frame size: 888
Unique wide receivers: 151
Teams represented: 32
Weeks represented: 18


In [10]:
print("Missing average separation:",
      q3_access_frame["avg_separation"].null_count())

print("Missing receiving yards:",
      q3_access_frame["yards"].null_count())

print("Missing targets:",
      q3_access_frame["targets"].null_count())

print("Missing average intended air yards:",
      q3_access_frame["avg_intended_air_yards"].null_count())

print(
    "Observations with zero targets:",
    q3_access_frame.filter(pl.col("targets") == 0).height
)

Missing average separation: 0
Missing receiving yards: 1
Missing targets: 0
Missing average intended air yards: 0
Observations with zero targets: 0


In [11]:
q3_sample = q3_access_frame.filter(
    pl.col("avg_separation").is_not_null() &
    pl.col("yards").is_not_null() &
    pl.col("targets").is_not_null() &
    (pl.col("targets") > 0) &
    pl.col("avg_intended_air_yards").is_not_null()
)

print("Access frame size:", q3_access_frame.height)
print("Final sample size:", q3_sample.height)

print(
    "Observations excluded:",
    q3_access_frame.height - q3_sample.height
)

Access frame size: 888
Final sample size: 887
Observations excluded: 1


In [12]:
teams_in_q3 = sorted(
    q3_access_frame["team_abbr"]
    .drop_nulls()
    .unique()
    .to_list()
)

print("Number of teams:", len(teams_in_q3))
print("Teams represented:", teams_in_q3)

Number of teams: 32
Teams represented: ['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LAC', 'LAR', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


# Part 3: Codebook Verification

In [13]:
def check_codebook(df, columns):
    results = []

    for col in columns:
        if col not in df.columns:
            print(f"Column not found: {col}")
            continue

        series = df[col]

        results.append({
            "column": col,
            "data_type": str(series.dtype),
            "missing_count": series.null_count(),
            "unique_values": series.n_unique(),
            "min_value": str(series.min()) if series.dtype.is_numeric() else "N/A",
            "max_value": str(series.max()) if series.dtype.is_numeric() else "N/A"
        })

    return pl.DataFrame(results)

In [14]:
q1_columns = [
    "season_type",
    "game_id",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "posteam",
    "play_type",
    "success"
]

check_codebook(playbyplay, q1_columns)

column,data_type,missing_count,unique_values,min_value,max_value
str,str,i64,i64,str,str
"""season_type""","""String""",0,2,"""N/A""","""N/A"""
"""game_id""","""String""",0,285,"""N/A""","""N/A"""
"""home_team""","""String""",0,32,"""N/A""","""N/A"""
"""away_team""","""String""",0,32,"""N/A""","""N/A"""
"""home_score""","""Int32""",0,42,"""0""","""52"""
"""away_score""","""Int32""",0,41,"""0""","""48"""
"""posteam""","""String""",2729,33,"""N/A""","""N/A"""
"""play_type""","""String""",1445,10,"""N/A""","""N/A"""
"""success""","""Float64""",570,3,"""0.0""","""1.0"""


In [15]:
q2_columns = [
    "season_type",
    "down",
    "play_type",
    "ydstogo",
    "yardline_100",
    "epa"
]

check_codebook(playbyplay, q2_columns)

column,data_type,missing_count,unique_values,min_value,max_value
str,str,i64,i64,str,str
"""season_type""","""String""",0,2,"""N/A""","""N/A"""
"""down""","""Float64""",7993,5,"""1.0""","""4.0"""
"""play_type""","""String""",1445,10,"""N/A""","""N/A"""
"""ydstogo""","""Float64""",0,37,"""0.0""","""36.0"""
"""yardline_100""","""Float64""",3548,100,"""1.0""","""99.0"""
"""epa""","""Float64""",570,43147,"""-12.658111390424892""","""7.967405086848885"""


In [16]:
import nflreadpy as nfl
import polars as pl

wr_receiving = nfl.load_nextgen_stats(
    seasons=2025,
    stat_type="receiving"
)

print(wr_receiving.shape)

wr_receiving = wr_receiving.filter(
    (pl.col("season_type") == "REG") &
    (pl.col("player_position") == "WR") &
    (pl.col("week") > 0)
)

print("WR observations:", wr_receiving.height)

q3_columns = [
    "player_gsis_id",
    "season",
    "week",
    "player_position",
    "team_abbr",
    "avg_separation",
    "avg_intended_air_yards",
    "yards",
    "targets",
    "receptions"
]

check_codebook(wr_receiving, q3_columns)

(1402, 23)
WR observations: 888


column,data_type,missing_count,unique_values,min_value,max_value
str,str,i64,i64,str,str
"""player_gsis_id""","""String""",0,151,"""N/A""","""N/A"""
"""season""","""Int32""",0,1,"""2025""","""2025"""
"""week""","""Int32""",0,18,"""1""","""18"""
"""player_position""","""String""",0,1,"""N/A""","""N/A"""
"""team_abbr""","""String""",0,32,"""N/A""","""N/A"""
"""avg_separation""","""Float64""",0,888,"""0.7163408198512767""","""6.945923061537053"""
"""avg_intended_air_yards""","""Float64""",0,887,"""-1.56""","""33.03"""
"""yards""","""Int32""",1,154,"""-2""","""225"""
"""targets""","""Int32""",0,15,"""5""","""23"""


# Part 4: Recipe

## Step One - Loading the Datasets

We loaded three datasets using the nflreadpy package:

NFL Play-by-Play Data (2025): Contains information about individual NFL plays, including teams, play types, scores, down, yards to go, and expected points added (EPA).

NFL Next Gen Stats Receiving Data (2025): Contains receiver statistics, including average separation, receiving yards, targets, and receptions.

NFL Weekly Player Statistics (2025): Contains weekly player performance statistics used to analyze receiving yards and targets.

The datasets were loaded directly through nflreadpy rather than manually downloading CSV files.


## Step Two - Filtering the Data

For our first research question, we used the 2025 NFL play-by-play dataset to identify all 32 NFL teams and their regular-season games.

For our second research question, we filtered the play-by-play dataset to include only regular-season fourth-down plays. We then kept plays classified as runs, passes, or punts. This reduced the dataset from 4,081 fourth-down plays to 2,815 relevant plays.

For our third research question, we filtered the Next Gen Stats receiving dataset to include only wide receivers during the 2025 regular season. This resulted in 888 player-week observations.

## Step Three - Cleaning the Data
After filtering the datasets, we checked for missing values in the columns needed for our research.

For Question 1, we filtered the play-by-play data to the 2025 regular season and selected rushing and passing plays for the team-level success-rate analysis.

For Question 2, we checked yards to go, EPA, and field position. No observations in our selected fourth-down sample were missing these values, so the sample remained at 2,815 plays.

For Question 3, we checked average separation, receiving yards, targets, and average intended air yards. One observation was missing receiving yards, so it was excluded. This left 887 observations in the scope-check sample.

## Step Four - Preparing the Data for Analysis
For Question 1, we filtered the play-by-play data to the 2025 regular season. We grouped plays by game to determine the winning team and calculated each team's win percentage. We then grouped rushing and passing plays by offensive team and play type to calculate average success rates. These results were joined by team to create the dataframe used in our scatterplot.

For Question 2, we filtered regular-season fourth-down plays to runs, passes, and punts. We created a new column called decision, which classifies punts as "Punt" and runs or passes as "Go for it." We then limited the analysis to situations with 1–10 yards to go, grouped the observations by yards to go and decision, and calculated average EPA. Groups with fewer than 10 plays were excluded.

For Question 3, we selected 2025 weekly player receiving statistics and joined them with Next Gen Stats receiving data. We removed observations with zero targets or missing receiving yards or average separation. We then created yards_per_target by dividing weekly receiving yards by weekly targets.

Code references: eda.ipynb, code cells under Relationship Plots 1, 2, and 3.

## Step Five - Dataset Joins
Our analysis includes two joins.

Question 1: Team Success Rates and Win Percentage

We joined team rushing and passing success rates with team win percentages.

Left join key: posteam

Right join key: team

Join type: Inner join (Polars default)

Purpose: Connect offensive performance with team winning percentage.

Matched observations: 64

Unmatched observations: 0 from success rates and 0 from win percentages.

All 32 NFL teams were successfully matched, with two observations per team representing rushing and passing success rates.

Question 3: Wide Receiver Statistics

We joined the 2025 Next Gen Stats receiving dataset with weekly NFL player statistics.

Join keys: player_gsis_id, season, and week

Join type: Inner join

Purpose: Match receiver separation measurements with receiving yards and targets for the same player and week.

Next Gen Stats observations before joining: 1,402

Weekly player statistics observations before joining: 19,422

Matched observations: 1,275

Unmatched observations: 127 from Next Gen Stats and 18,147 from weekly player statistics, totaling 18,274 unmatched observations.

After joining and applying the final cleaning filters, 1,275 observations remained for the relationship plot.
Code references: eda.ipynb, Relationship Plot 1 and Relationship Plot 3.

## Step Six - Reproducing the Results
To reproduce our data preparation and analysis, another user can clone our GitHub repository and open datasheet_checks.ipynb and eda.ipynb.

The required Python packages can be installed using:

python -m pip install polars nflreadpy numpy pandas matplotlib seaborn plotly pyarrow jupyter

First, run datasheet_checks.ipynb to load the datasets, apply the filters, check missing values, and reproduce the scope checks.

The expected sample sizes are:

Question 1: All 32 NFL teams are represented.

Question 2: 2,815 fourth-down plays.

Question 3: 887 receiver observations after the initial missing-value checks.

Next, run eda.ipynb to reproduce the final data preparation, dataset joins, relationship plots, and analysis.

# Part 5: FAIR Principles
We use the FAIR (Findable, accessible, interoperable, reusable) priniciples in this project

# Findable
We used NFL play-by-play data, NFL weekly player statistics, and NFL Next Gen Stats receiving data. These datasets were accessed through the nflreadpy Python package.

# Accessible
The datasets can be loaded using Python and nflreadpy. This allows other researchers to retrieve the same types of data without manually downloading files.

# Interoperable
We used Python and Polars to process the datasets. The data is stored in tabular form, making it possible to filter, clean, combine, and analyze the observations.

# Reusable
Our GitHub repository contains the code used to prepare and analyze the datasets. Other researchers can use this code to reproduce our analysis.

Reuse and redistribution of the underlying NFL data depend on the original publishers' licenses and terms of use. We must verify those terms before claiming that our processed datasets can be redistributed.

### Dataset Access and Reuse Information

| Dataset | Source Link | Access Date | File Format | License / Reuse |
|---|---|---|---|---|
| NFL Play-by-Play (2025) | [nflverse](https://github.com/nflverse/nflverse-data) | Oct. 28, 2026 | Parquet available | CC BY 4.0 |
| NFL Weekly Player Stats | [nflverse](https://github.com/nflverse/nflverse-data) | Oct. 28, 2026 | Parquet available | CC BY 4.0 |
| NFL Next Gen Stats Receiving | [nflverse](https://github.com/nflverse/nflverse-data) | Oct. 28, 2026 | Parquet available | CC BY 4.0 |

Our project shares the code needed to reproduce our processed datasets rather than redistributing the original NFL data. Anyone using our code must follow the original data publishers' terms of use.

# Part 6: CARE Principles